# How to Handle Missing Values in Data Engineering (Spark / PySpark / SQL / Hive)

Missing values (NULLs) are very common in real-time data pipelines due to source system issues, optional fields, API failures, or data integration problems. Handling them correctly is critical for accurate reporting and analytics.

---

# 1. Drop Missing Values (Remove Records)

## When to use
- Missing value is in a **mandatory field (business key, primary key)**
- Record is unusable without that field

## PySpark

```python
df.na.drop()
```

Drop based on specific columns:

```python
df.na.drop(subset=["customer_id", "email"])
```

## SQL / Hive

```sql
SELECT *
FROM customer
WHERE customer_id IS NOT NULL;
```

## Real-time example
- Orders without `order_id`
- Transactions without `transaction_id`

---

# 2. Fill Missing Values with Default Values

## When to use
- Optional fields
- Business-defined default values

## PySpark

```python
df.na.fill("Unknown")
df.na.fill(0)
```

Column-wise fill:

```python
df.na.fill({
    "city": "Unknown",
    "salary": 0
})
```

## SQL / Hive

```sql
SELECT COALESCE(city, 'Unknown') AS city
FROM employee;
```

## Real-time example
- Missing city → "Unknown"
- Missing salary → 0

---

# 3. Using COALESCE (Best Practice in SQL / Spark SQL)

## When to use
- Multiple fallback values
- Preferred ANSI SQL approach

## SQL

```sql
SELECT COALESCE(phone, mobile, 'NA')
FROM customer;
```

## PySpark

```python
from pyspark.sql.functions import coalesce

df = df.withColumn("contact", coalesce("phone", "mobile"))
```

## Real-time example
- Use mobile if phone is missing

---

# 4. Conditional Handling (CASE WHEN)

## When to use
- Business-specific logic

## SQL

```sql
SELECT
CASE
    WHEN salary IS NULL THEN 0
    ELSE salary
END AS salary
FROM employee;
```

## PySpark

```python
from pyspark.sql.functions import when, col

df = df.withColumn(
    "salary",
    when(col("salary").isNull(), 0).otherwise(col("salary"))
)
```

---

# 5. Replace with Mean / Median (Statistical Imputation)

## When to use
- Machine Learning / analytics use cases
- Numerical columns

## PySpark

```python
mean_salary = df.selectExpr("avg(salary)").first()[0]

df = df.na.fill(mean_salary, ["salary"])
```

## Real-time example
- Predicting missing income values in ML models

---

# 6. Forward Fill (Time Series Data)

## When to use
- Stock data
- Sensor data
- Time-based data

## PySpark

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import last

window = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

df = df.withColumn(
    "value",
    last("value", True).over(window)
)
```

---

# 7. Backward Fill

## When to use
- Fill from next available value

## PySpark

```python
from pyspark.sql.functions import first

window = Window.orderBy("date").rowsBetween(0, Window.unboundedFollowing)

df = df.withColumn(
    "value",
    first("value", True).over(window)
)
```

---

# 8. Handle NULLs During Joins

## When to use
- LEFT JOIN creates NULLs

## SQL

```sql
SELECT
a.id,
COALESCE(b.salary, 0) AS salary
FROM emp a
LEFT JOIN salary b
ON a.id = b.id;
```

## Real-time example
- Missing dimension records in fact tables

---

# 9. Convert Empty Strings to NULL

## When to use
- Source systems send "" instead of NULL

## PySpark

```python
from pyspark.sql.functions import when, col

df = df.withColumn(
    "city",
    when(col("city") == "", None).otherwise(col("city"))
)
```

---

# 10. Drop or Fill Based on Business Rules

## When to use
- Depends on field importance

### Drop (mandatory fields)

```python
df = df.na.drop(subset=["customer_id"])
```

### Fill (optional fields)

```python
df = df.na.fill("Unknown")
```

---

# 11. Data Quality Check for NULLs

## PySpark

```python
from pyspark.sql.functions import col, sum, when

df.select(
    sum(when(col("salary").isNull(), 1).otherwise(0)).alias("null_count")
)
```

---

# 12. NULL in Aggregations

Aggregations ignore NULL values by default.

## SQL

```sql
SELECT AVG(salary)
FROM employee;
```

To treat NULL as 0:

```sql
SELECT AVG(COALESCE(salary, 0))
FROM employee;
```

---

# Real-Time Strategy Summary

| Scenario | Best Approach |
|----------|--------------|
| Missing primary key | Drop record |
| Optional string field | Fill "Unknown" |
| Numeric reporting field | Fill 0 |
| ML feature | Mean/Median imputation |
| Time series | Forward fill |
| Join results | COALESCE |
| Empty strings | Convert to NULL first |
| Business logic cases | CASE WHEN |

---

# Key Takeaway

Handling missing values depends on **business requirements, data type, and use case**:

- Drop records when data is critical
- Fill values when business-defined defaults exist
- Use COALESCE for SQL-based pipelines
- Use statistical methods for ML models
- Use forward/backward fill for time series data

---

# Best Practices

- Always understand business meaning before filling NULLs
- Never blindly replace missing values
- Track NULL counts for data quality monitoring
- Handle NULLs early in ETL pipelines (Bronze → Silver layer)
- Use COALESCE in SQL-based transformations
```